# XGBoost sliding-window churn (future 10 days)

滑动窗口示例：每日为锚点，未来 10 天是否出现 `Cancellation Confirmation` 为标签。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

from feature_pipeline import feature_engineer, encode_categoricals, _build_mapping, _infer_numeric_cols

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "churn-prediction-25-26"
TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"
OUTPUT_DIR = Path("outputs_xgb_sliding")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZON_DAYS = 10
ANCHOR_END = pd.Timestamp("2018-11-09")
VAL_FRAC = 0.2

In [ ]:
def prepare_event_features(train_path: Path, test_path: Path):
    train_raw = pd.read_parquet(train_path)
    test_raw = pd.read_parquet(test_path)
    label_raw = train_raw.copy()
    train_no_churn = train_raw[train_raw["page"] != "Cancellation Confirmation"].copy()

    page_categories = sorted(set(train_no_churn["page"]).union(set(test_raw["page"])))
    train_fe = feature_engineer(train_no_churn, page_categories)
    test_fe = feature_engineer(test_raw, page_categories)

    metro_map = _build_mapping([train_fe["metro"], test_fe["metro"]])
    state_map = _build_mapping([train_fe["state"], test_fe["state"]])
    device_map = _build_mapping([train_fe["device"], test_fe["device"]])

    train_fe = encode_categoricals(train_fe, page_categories, metro_map, state_map, device_map)
    test_fe = encode_categoricals(test_fe, page_categories, metro_map, state_map, device_map)

    for df in (train_fe, test_fe):
        df["prev_page_id"] = df.groupby("userId")["page_id"].shift(1).fillna(0).astype(np.int64)
        if "error occur" in df.columns:
            df.rename(columns={"error occur": "error_occur"}, inplace=True)
        df["regis_time_seconds"] = df["regis_time"].dt.total_seconds()

    drop_cols = {"regis_time", "song", "artist", "page", "metro", "state", "device"}
    train_fe = train_fe.drop(columns=[c for c in drop_cols if c in train_fe.columns])
    test_fe = test_fe.drop(columns=[c for c in drop_cols if c in test_fe.columns])

    numeric_cols = sorted(set(_infer_numeric_cols(train_fe) + ["regis_time_seconds"]))
    for df in (train_fe, test_fe):
        for col in numeric_cols:
            if col not in df:
                df[col] = 0.0
        df[numeric_cols] = df[numeric_cols].fillna(0)

    cat_id_cols = ["page_id", "metro_id", "state_id", "device_id", "prev_page_id"]
    return train_fe, test_fe, label_raw, numeric_cols, cat_id_cols


def aggregate_window(window: pd.DataFrame, numeric_cols, cat_id_cols):
    stats = {}
    stats["event_count"] = float(len(window))
    if len(window) > 1:
        span_seconds = (window["time"].iloc[-1] - window["time"].iloc[0]).total_seconds()
    else:
        span_seconds = 0.0
    stats["active_hours"] = span_seconds / 3600.0
    stats["events_per_hour"] = stats["event_count"] / max(stats["active_hours"], 1e-3)

    gap_seconds = window["time"].diff().dt.total_seconds().dropna()
    stats["median_gap_seconds"] = float(gap_seconds.median()) if not gap_seconds.empty else 0.0
    stats["mean_gap_seconds"] = float(gap_seconds.mean()) if not gap_seconds.empty else 0.0

    stats["days_since_registration_last"] = window["regis_time_seconds"].iloc[-1] / 86400.0
    stats["paid_ratio"] = window["level"].mean()
    stats["paid_last"] = float(window["level"].iloc[-1])
    stats["level_switches"] = float(window["level"].diff().fillna(0).abs().sum())

    for col in cat_id_cols:
        stats[f"{col}_last"] = float(window[col].iloc[-1])
        stats[f"{col}_nunique"] = float(window[col].nunique())

    for col in numeric_cols:
        series = window[col]
        stats[f"{col}_mean"] = float(series.mean())
        stats[f"{col}_std"] = float(series.std(ddof=0))
        stats[f"{col}_max"] = float(series.max())
        stats[f"{col}_last"] = float(series.iloc[-1])

    return stats


def build_sliding_tabular(feat_df: pd.DataFrame, label_raw: pd.DataFrame, numeric_cols, cat_id_cols, horizon_days: int, anchor_end: pd.Timestamp | None):
    horizon = pd.Timedelta(days=horizon_days)
    churn_times = (
        label_raw[label_raw["page"] == "Cancellation Confirmation"]
        .groupby("userId")["time"]
        .apply(list)
        .to_dict()
    )

    rows = []
    labels = {}

    grouped = feat_df.sort_values(["userId", "time"]).groupby("userId", sort=False)
    for uid, g in grouped:
        if g.empty:
            continue
        last_valid = (g["time"].max() - horizon).normalize()
        days = pd.to_datetime(g["time"].dt.normalize().unique())
        for day in days:
            if day > last_valid:
                continue
            if anchor_end is not None and day > anchor_end:
                continue
            cutoff = day + pd.Timedelta(days=1)
            window = g[g["time"] < cutoff]
            if window.empty:
                continue
            sample_id = f"{uid}__{day.date()}"
            label = 0
            for t in churn_times.get(uid, []):
                if day < t <= day + horizon:
                    label = 1
                    break
            labels[sample_id] = label
            stats = aggregate_window(window, numeric_cols, cat_id_cols)
            stats.update({"sample_id": sample_id, "userId": uid, "anchor_date": day})
            rows.append(stats)

    df = pd.DataFrame(rows)
    return df, labels


def build_test_tabular(feat_df: pd.DataFrame, numeric_cols, cat_id_cols, horizon_days: int, anchor_end: pd.Timestamp | None):
    horizon = pd.Timedelta(days=horizon_days)
    rows = []
    grouped = feat_df.sort_values(["userId", "time"]).groupby("userId", sort=False)
    for uid, g in grouped:
        if g.empty:
            continue
        candidate = (g["time"].max() - horizon).normalize()
        anchor_day = candidate if anchor_end is None else min(candidate, anchor_end)
        cutoff = anchor_day + pd.Timedelta(days=1)
        window = g[g["time"] < cutoff]
        if window.empty:
            window = g
        stats = aggregate_window(window, numeric_cols, cat_id_cols)
        stats.update({"sample_id": f"{uid}__test", "userId": uid, "anchor_date": anchor_day})
        rows.append(stats)

    return pd.DataFrame(rows)


def time_based_split(df: pd.DataFrame, labels: dict, val_frac: float = 0.2):
    anchor_dates = pd.to_datetime(df["anchor_date"])
    split_date = anchor_dates.quantile(1 - val_frac)
    train_mask = anchor_dates <= split_date
    train_df = df[train_mask].reset_index(drop=True)
    val_df = df[~train_mask].reset_index(drop=True)
    y_train = train_df["sample_id"].map(labels).astype(int)
    y_val = val_df["sample_id"].map(labels).astype(int)
    return train_df, val_df, y_train, y_val


train_fe, test_fe, label_raw, numeric_cols, cat_id_cols = prepare_event_features(TRAIN_PATH, TEST_PATH)
train_tab, labels = build_sliding_tabular(train_fe, label_raw, numeric_cols, cat_id_cols, HORIZON_DAYS, ANCHOR_END)
train_df, val_df, y_train, y_val = time_based_split(train_tab, labels, VAL_FRAC)

feature_cols = [c for c in train_df.columns if c not in {"sample_id", "userId", "anchor_date"}]
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
val_df[feature_cols] = scaler.transform(val_df[feature_cols])

print(f"Train samples: {len(train_df)}, Val samples: {len(val_df)}, Pos rate: {np.mean(list(labels.values())):.3f}")

In [ ]:
pos = float(y_train.sum())
neg = float(len(y_train) - pos)
pos_weight = neg / max(pos, 1.0)

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=1.0,
    reg_lambda=1.0,
    reg_alpha=0.0,
    gamma=0.0,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=8,
    random_state=42,
    scale_pos_weight=pos_weight,
)

if len(val_df) > 0:
    model.fit(train_df[feature_cols], y_train, eval_set=[(val_df[feature_cols], y_val)], verbose=False)
else:
    model.fit(train_df[feature_cols], y_train)

val_logloss = None
if len(val_df) > 0:
    val_pred = model.predict_proba(val_df[feature_cols])[:, 1]
    val_logloss = log_loss(y_val, val_pred)
print(f"Val logloss: {val_logloss}")

model.save_model(OUTPUT_DIR / "xgb_sliding.json")

In [ ]:
test_tab = build_test_tabular(test_fe, numeric_cols, cat_id_cols, HORIZON_DAYS, ANCHOR_END)
test_tab[feature_cols] = scaler.transform(test_tab[feature_cols])
test_pred = model.predict_proba(test_tab[feature_cols])[:, 1]

submission = pd.DataFrame({
    "id": test_tab["userId"],
    "target": (test_pred >= 0.5).astype(int),
    "probability": test_pred,
})
submission_path = OUTPUT_DIR / "xgb_sliding_submission.csv"
submission.to_csv(submission_path, index=False)
submission.head()